In [1]:
import pandas as pd
from pandasql import sqldf
run_sql = lambda q: sqldf(q, globals())

# Create the Movies table
# Note: The tied ratings in Sci-Fi and Drama
movies_data = {
    'movie_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'title': ['The Sci-Fi Epic', 'Space Adventure', 'Cosmic Journey',
              'Laugh Riot', 'Funny Business', 'The Drama King',
              'Tearjerker', 'Another Drama', 'Action Hero', 'Thriller Night'],
    'genre': ['Sci-Fi', 'Sci-Fi', 'Sci-Fi', 'Comedy', 'Comedy',
              'Drama', 'Drama', 'Drama', 'Action', 'Thriller'],
    'rating': [9.2, 8.5, 9.2, 7.8, 6.5, 9.0, 8.1, 9.0, 7.5, 8.8]
}
Movies = pd.DataFrame(movies_data)

print("--- Movies Table ---")
print(Movies)
print("\nSetup complete! You have one table: 'Movies'.")

--- Movies Table ---
   movie_id            title     genre  rating
0         1  The Sci-Fi Epic    Sci-Fi     9.2
1         2  Space Adventure    Sci-Fi     8.5
2         3   Cosmic Journey    Sci-Fi     9.2
3         4       Laugh Riot    Comedy     7.8
4         5   Funny Business    Comedy     6.5
5         6   The Drama King     Drama     9.0
6         7       Tearjerker     Drama     8.1
7         8    Another Drama     Drama     9.0
8         9      Action Hero    Action     7.5
9        10   Thriller Night  Thriller     8.8

Setup complete! You have one table: 'Movies'.


In [2]:
# Query 1: Use a CTE to select only the high-rated movies
query1 = """
WITH HighRatedMovies AS (
    SELECT
        title,
        genre,
        rating
    FROM
        Movies
    WHERE
        rating > 8.0
)
SELECT *
FROM HighRatedMovies
WHERE genre = 'Sci-Fi';
"""

results1 = run_sql(query1)
print(results1)

             title   genre  rating
0  The Sci-Fi Epic  Sci-Fi     9.2
1  Space Adventure  Sci-Fi     8.5
2   Cosmic Journey  Sci-Fi     9.2


In [3]:
# Query 2: Rank movies within their genre based on rating
query2 = """
SELECT
    title,
    genre,
    rating,
    RANK() OVER (PARTITION BY genre ORDER BY rating DESC) AS genre_rank
FROM
    Movies;
"""

results2 = run_sql(query2)
print(results2)

             title     genre  rating  genre_rank
0      Action Hero    Action     7.5           1
1       Laugh Riot    Comedy     7.8           1
2   Funny Business    Comedy     6.5           2
3   The Drama King     Drama     9.0           1
4    Another Drama     Drama     9.0           1
5       Tearjerker     Drama     8.1           3
6  The Sci-Fi Epic    Sci-Fi     9.2           1
7   Cosmic Journey    Sci-Fi     9.2           1
8  Space Adventure    Sci-Fi     8.5           3
9   Thriller Night  Thriller     8.8           1


In [4]:
# Query 3: Use a CTE to find the top 3 movies per genre
query3 = """
WITH RankedMovies AS (
    SELECT
        title,
        genre,
        rating,
        ROW_NUMBER() OVER (PARTITION BY genre ORDER BY rating DESC) AS genre_rank
    FROM
        Movies
)
SELECT *
FROM RankedMovies
WHERE genre_rank <= 3;
"""

results3 = run_sql(query3)
print(results3)

             title     genre  rating  genre_rank
0      Action Hero    Action     7.5           1
1       Laugh Riot    Comedy     7.8           1
2   Funny Business    Comedy     6.5           2
3   The Drama King     Drama     9.0           1
4    Another Drama     Drama     9.0           2
5       Tearjerker     Drama     8.1           3
6  The Sci-Fi Epic    Sci-Fi     9.2           1
7   Cosmic Journey    Sci-Fi     9.2           2
8  Space Adventure    Sci-Fi     8.5           3
9   Thriller Night  Thriller     8.8           1


In [5]:
# Query 4: Compare each movie's rating to the average rating of its genre
query4 = """
SELECT
    title,
    genre,
    rating,
    AVG(rating) OVER (PARTITION BY genre) AS avg_genre_rating
FROM
    Movies;
"""

results4 = run_sql(query4)
print(results4)

             title     genre  rating  avg_genre_rating
0      Action Hero    Action     7.5          7.500000
1       Laugh Riot    Comedy     7.8          7.150000
2   Funny Business    Comedy     6.5          7.150000
3   The Drama King     Drama     9.0          8.700000
4       Tearjerker     Drama     8.1          8.700000
5    Another Drama     Drama     9.0          8.700000
6  The Sci-Fi Epic    Sci-Fi     9.2          8.966667
7  Space Adventure    Sci-Fi     8.5          8.966667
8   Cosmic Journey    Sci-Fi     9.2          8.966667
9   Thriller Night  Thriller     8.8          8.800000
